# La Grulla Blanca — Definitivas Fuego + Metal + Agua
Benchmark mecánico pesado sobre `NEW_COMBAT_STATS_V0_1`. No es runtime ni cierre de balance.

Reconstruye desde GitHub el paquete base verificado y aplica el override **v0.2-followup-chains** también verificado por partes.

Se prueban activación, cadenas posteriores, consumo de marcas/ventanas, clímax y expiraciones.

**Salida para devolver a ChatGPT:** `ULTI_3F_RESULTS_FOR_CHATGPT.zip`.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, urllib.parse, urllib.request, zipfile

OWNER = "Shein25"
REPO = "La-Grulla-Blanca"
BRANCH = "experiment/ultis-kaggle-3familias-v0.1"
BASE_ROOT = "experimentos/ultis/kaggle/package_chunks_v2"
BASE_SHA256 = "18314ab1614d930b7fce22919a3fc5c63c846980c9fc0fe5222eaefb2aa29744"
OVERRIDE_ROOT = "experimentos/ultis/kaggle/followup_override_v02_parts"
OVERRIDE_SHA256 = "3d7c011b5ca5f6433ebf952d617c5f8d99017f30733a378e67ce3c3941181abf"

def github_text(path):
    ref = urllib.parse.quote(BRANCH, safe="")
    path_q = "/".join(urllib.parse.quote(x, safe="") for x in path.split("/"))
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{path_q}?ref={ref}"
    req = urllib.request.Request(url, headers={"Accept":"application/vnd.github+json","User-Agent":"La-Grulla-Blanca-Kaggle-LAB/1.2"})
    with urllib.request.urlopen(req, timeout=60) as r:
        obj = json.loads(r.read().decode("utf-8"))
    if obj.get("encoding") != "base64":
        raise RuntimeError(f"Contenido GitHub inesperado para {path}: {obj.get('encoding')}")
    return base64.b64decode(obj["content"]).decode("utf-8")

def reconstruct_from_manifest(root, expected_sha):
    manifest = json.loads(github_text(f"{root}/manifest.json"))
    joined = "".join(github_text(f"{root}/{name}").strip() for name in manifest["parts"])
    if len(joined) != manifest["base64_length"]:
        raise RuntimeError(f"Base64 incompleto en {root}: {len(joined)} != {manifest['base64_length']}")
    data = base64.b64decode(joined, validate=True)
    sha = hashlib.sha256(data).hexdigest()
    if len(data) != manifest["decoded_size_bytes"]:
        raise RuntimeError(f"Tamaño incorrecto en {root}: {len(data)} != {manifest['decoded_size_bytes']}")
    if sha != expected_sha or sha != manifest["sha256"]:
        raise RuntimeError(f"SHA-256 incorrecto en {root}: {sha}")
    return manifest, data, sha

# 1) Paquete base estable ya publicado.
base_manifest, base_data, base_sha = reconstruct_from_manifest(BASE_ROOT, BASE_SHA256)
ROOT = Path("/kaggle/working/ulti3f_pkg")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
base_zip = Path("/kaggle/working") / base_manifest["package"]
base_zip.write_bytes(base_data)
with zipfile.ZipFile(base_zip) as z:
    bad=z.testzip()
    if bad: raise RuntimeError(f"ZIP base corrupto: {bad}")
    z.extractall(ROOT)

# 2) Override v0.2: cadenas posteriores de las Ultis.
override_manifest, override_data, override_sha = reconstruct_from_manifest(OVERRIDE_ROOT, OVERRIDE_SHA256)
override_zip = Path("/kaggle/working") / override_manifest["package"]
override_zip.write_bytes(override_data)
with zipfile.ZipFile(override_zip) as z:
    bad=z.testzip()
    if bad: raise RuntimeError(f"Override corrupto: {bad}")
    z.extractall(ROOT)

catalog=json.loads((ROOT/"ultimates_catalog_v0_1.json").read_text())
if not catalog.get("global_rules",{}).get("followup_chain_effects_are_tested"):
    raise RuntimeError("El override no activó la auditoría de cadenas posteriores")

print("GIT_PACKAGE_OK")
print("Branch:", BRANCH)
print("Base SHA-256:", base_sha)
print("Override SHA-256:", override_sha)
print("Package efectivo: v0.2-followup-chains")
print("Runner:", ROOT/"run_ultimates_3families_kaggle.py")


In [ ]:
# Smoke obligatorio antes de la corrida pesada.
SMOKE_OUT = Path('/kaggle/working/smoke_ulti_3f')
cmd = f"python {ROOT/'run_ultimates_3families_kaggle.py'} --mode smoke --runs 200 --workers 0 --output {SMOKE_OUT}"
print(cmd)
rc=os.system(cmd)
if rc != 0: raise RuntimeError(f"Smoke falló con código {rc}")
selfcheck=json.loads((SMOKE_OUT/'SELF_CHECK.json').read_text())
if not selfcheck.get('pass'): raise RuntimeError(f"SELF_CHECK falló: {selfcheck}")
print('SMOKE + SELF_CHECK: PASS')


In [ ]:
# Corrida pesada: 45 escenarios x 100.000 corridas.
RUNS = 100_000
OUT = Path('/kaggle/working/results_ulti_3f')
cmd = f"python {ROOT/'run_ultimates_3families_kaggle.py'} --mode heavy --runs {RUNS} --workers 0 --output {OUT}"
print(cmd)
rc=os.system(cmd)
if rc != 0: raise RuntimeError(f'Runner terminó con código {rc}')


In [ ]:
from IPython.display import FileLink, display
result=OUT/'ULTI_3F_RESULTS_FOR_CHATGPT.zip'
print('Resultado compacto:', result, f'{result.stat().st_size/1024:.1f} KiB')
display(FileLink(str(result)))
print('Devuelve únicamente ese ZIP a ChatGPT.')
